# End-to-End Workflow: New RAND HRS Use Case

### Overview

#### Goal: 
Deliver governed, trustworthy RAND HRS analyses in the Silver CDM — from an end user's business question to loaded, verified warehouse tables and consumable gold-layer results — through a repeatable, auditable process with clear human approval gates and a controlled AI drafting step.

#### Purpose:

* `Governance by design`. Every variable that enters the warehouse is traceable to an approved Business Use Case Document, an approved-variable dictionary entry, a bronze source match, and a technical sign-off — so "why is this column here?" always has an answer, and end users authorize intent before any data work begins.
* `Reuse over rebuild`. The 5-entity Silver CDM (hub/dimension/fact pattern) is the single home for person-wave HRS data. New use cases route against the live schema first — variables with existing homes produce a coverage report, never duplicate tables. Only genuine gaps generate new DDL/DML.
* `Human-machine separation of duties`. The agent scopes, routes, and drafts (REVIEW_ONLY, grounded solely on approved metadata for the named use case); humans alone approve, commit, and execute. No write ever happens without an explicit human action.
* `Per-use-case isolation`. Each use case's dictionary scopes the agent's entire grounding and coverage — analyses never see each other's variables — while shared warehouse columns serve them all, keeping the model idempotent as use cases accumulate.  In other words, `idempotent` means: no matter how many times you repeat the operation — or in what order — you end up with the same result. Applying it twice doesn't produce two of something.  In this workflow specifically, `idempotent` phrase refers to the fact that use cases can share the warehouse without ever duplicating it, because the agent routes against the live information schema before drafting anything:
    * Use case A needed R{wave}HEIGHT → gap found → draft → you deployed fact_health.height.

    * Use case B also needs R{wave}HEIGHT → the agent checks the live schema, sees height already exists, and produces a coverage report — not a second column, a second table, or a duplicate draft.
    * Run use case B a third time, or reverse the order of A and B — same outcome every time.

    So the warehouse converges to one state regardless of how many use cases run, how often they're re-run, or which one ran first. Each variable family gets exactly one home, period.
* `Repeatability`. Notebook widgets (1.5.2, 1.5.3, Run HRS Agent), parameterized agent code, and repo-committed SQL make each new use case a configuration exercise — dictionary → approval → coverage → agent run → deploy/verify — not a code-change exercise.
In one sentence: the workflow converts an end user's business question into approved metadata, uses the agent to determine what (if anything) the warehouse is missing and draft the fix, and lands verified, human-executed changes in the Silver CDM — with an audit trail from business intent to loaded table.


# End-to-End Workflow

| Phase | # | Step | Role | Activity | Output / Artifact |
|---------|---|---------|---------|---------|---------|
| `Requirements` |  |  |  |  |  |
| 1. Requirements | 1 | Submit Request | End User | State the business question and scope. | Request |
| 1. Requirements | 2 | Analysis & Data Requirements | Data Analyst | Identify RAND variables, analysis grain, cohort scope, and existing Silver CDM coverage. | Requirements Summary |
| 1. Requirements | 3 | Create Business Use Case Document | Data Analyst | Create the use case name, description, required variables, grain, wave coverage, and expected routing outcomes. | Business Use Case Document |
| 1. Requirements | 4 | Review & Approve | End User | Review and approve the Business Use Case Document. | Approved Use Case |
| `Foundation` |  |  |  |  |  |
| 2. Foundation | 5 | Foundation Notebooks (1.1–1.4.2) | Data Engineer | Run table extraction, Codebook page processing, variable candidates, and bronze-column matching. Re-run 1.2–1.4.2 only when a new Codebook version is introduced. | `hrs_document_pages`, candidate/review chain |
| 3. Dictionary & Approval | 6 | Curate Dictionary (1.5.1) | Data Analyst | Create a use-case-specific dictionary containing only the variables required by the use case. Verify labels against the Codebook and confirm bronze column mappings. | `approved_variable_dictionary_b` |
| 3. Dictionary & Approval | 7 | Approval Processing (1.5.2) | Data Analyst | Configure `use_case_name` and `codebook_version` widgets and execute approval processing. Validate expected variable counts and bronze matches. | `hrs_use_case_approved_variables` |
| 3. Dictionary & Approval | 8 | Cohort Coverage (1.5.3) | Data Analyst | Execute cohort coverage evaluation using the same widget values and review `HAS_DATA` and `NO_DATA` results. | `hrs_use_case_approved_cohort_coverage` |
| 3. Dictionary & Approval | 9 | Technical Sign-Off | Data Analyst | Record `reviewed_by`, `reviewed_at`, and `wave_coverage_status` via UPDATE statements and validate against the Business Use Case Document. | Signed-Off Metadata |
| 3. Dictionary & Approval | 10 | Coverage Confirmation | End User | Confirm that `NO_DATA` implications are acceptable for analysis. | Acknowledged Coverage |
| `Agent Drafting` |  |  |  |  |  |
| 4. Agent Drafting | 11 | Set Use-Case Widgets | Data Analyst | Set `use_case_name` in notebooks 1.5.2, 1.5.3, and Run HRS Agent. Ensure `codebook_version` values match. Widget values persist between runs and should change only when switching use cases. | Configured Notebooks |
| 4. Agent Drafting | 12 | Open & Run HRS Agent | Data Engineer | Set the notebook `use_case_name` widget and execute Run All. Cell 1 scopes the agent. Cell 2 validates grounding and asserts that dictionary, approval, and coverage data are present. | Scoped and Verified Agent Session |
| 4. Agent Drafting | 13 | Read Routing Report (Cell 3) | Data Engineer | Review deterministic routing results against the live schema. If gaps are found, DDL/DML drafts are required; if no gaps exist, deployment may be skipped. | Routing Decision |
| 4. Agent Drafting | 14 | Agent Run (Cell 4) | Agent | Generate coverage reports, routing recommendations, DDL/DML drafts, and Codebook label drafts where required. Agent remains review-only and never executes changes. | Draft Artifacts and Audit Trail |
| 4. Agent Drafting | 15 | Review Drafts | Data Engineer | Review generated drafts against data type standards, grain requirements, and review checklist. Accept or reject each draft. | Draft Decisions |
| `Deploy & Verify` |  |  |  |  |  |
| 5. Deploy & Verify | 16 | Apply Drafts | Data Engineer | Move approved DDL/DML into repository scripts and complete feature branch → PR → CI → merge workflow. | Committed Scripts |
| 5. Deploy & Verify | 17 | Execute DDL + DML | Data Engineer | Execute repository-based scripts (never pasted SQL) against `dev_catalog`. Run DDL before DML. | Updated Silver Tables |
| 5. Deploy & Verify | 18 | Re-Run Agent Routing | Data Engineer → Agent | Re-run Run HRS Agent (Cells 2–4) to validate that all routing gaps have been resolved. | Zero-Gap Routing Report |
| 5. Deploy & Verify | 19 | Verify Loaded Tables | Data Engineer | Validate row counts, grain consistency, population rates, value distributions, and foreign key integrity. | Verification Results |
| 5. Deploy & Verify | 20 | Consume Results | End User | Use Gold facts, reports, dashboards, and analytics. Feedback may generate new requirements and restart the lifecycle. | Delivered Analysis |
| `Gold Design` |  |  |  |  |  |
| 6. Gold Design | 21 | Define Gold Deliverables | Data Analyst | Define required Gold-level aggregates, grain, dimensions, and display labels from the Use Case Document. | Gold Design Specification |
| 6. Gold Design | 22 | Check Existing Gold Coverage | Data Engineer | Determine whether existing Gold assets can support the use case before creating new structures. | Build vs. Reuse Decision |
| `Gold Drafting` |  |  |  |  |  |
| 7. Gold Drafting | 23 | Agent Gold Routing Run | Data Engineer → Agent | Generate Gold coverage reports and draft aggregate DDL/DML artifacts where gaps exist. | Coverage Report or Draft Artifacts |
| 7. Gold Drafting | 24 | Review Gold Drafts | Data Engineer | Validate aggregate design standards, joins, filtering, and presentation requirements. | Draft Decisions |
| 7. Gold Drafting | 25 | Codebook Label Cross-Check | Agent | Validate report-facing labels against approved Codebook definitions. | Label Verification |
| `Gold Deploy & Verify` |  |  |  |  |  |
| 8. Gold Deploy & Verify | 26 | Apply Drafts | Data Engineer | Commit approved Gold DDL/DML scripts through the Git workflow. | Committed Scripts |
| 8. Gold Deploy & Verify | 27 | Execute DDL + DML | Data Engineer | Execute Gold-layer aggregate loading processes from repository files. | Gold Facts Loaded |
| 8. Gold Deploy & Verify | 28 | Verify Aggregates | Data Engineer | Independently validate Gold aggregations and label mappings. | Verification Results |
| 8. Gold Deploy & Verify | 29 | Consume Results | End User | Use dashboards and reports based on Gold data products. | Delivered Analysis |

---

# Role Boundaries

| Role | Owns | Never Does |
|---------|---------|---------|
| End User | Requirements, Business Use Case approval, coverage confirmation, and consumption of analytical results. | N/A |
| Data Analyst | Business Use Case Document, dictionary creation, approval processing, coverage processing, and technical sign-off. | Execute DDL/DML or commit code. |
| Data Engineer | Agent execution, draft review, Git workflow, DDL/DML execution, deployment, and verification activities. | Approve business requirements. |
| Agent | Scoping, routing, coverage reporting, and DDL/DML draft generation based only on approved metadata for the selected use case. | Execute SQL, modify the warehouse, commit code, or read the Business Use Case Document. |

---

# Phase 1 – Requirements

| # | Step | Role | Activity | Output |
|---|---|---|---|---|
| 1 | Submit Request | End User | State the business question and scope. | Request |
| 2 | Analysis & Data Requirements | Data Analyst | Identify RAND variables, grain, cohort scope, and Silver CDM coverage. | Requirements Summary |
| 3 | Create Business Use Case Document | Data Analyst | Document use case details, required variables, grain, wave coverage, and expected routing outcomes. | Business Use Case Document |
| 4 | Review & Approve | End User | Review and approve the Business Use Case Document. | Approved Use Case |

---

# Phase 2 – Foundation

| # | Step | Role | Activity | Output |
|---|---|---|---|---|
| 5 | Foundation Notebooks (1.1–1.4.2) | Data Engineer | Execute Codebook processing, candidate generation, review tables, and bronze-column matching. | `hrs_document_pages`, candidate/review chain |

---

# Phase 3 – Dictionary & Approval

| # | Step | Role | Activity | Output |
|---|---|---|---|---|
| 6 | Curate Dictionary (1.5.1) | Data Analyst | Create use-case-specific dictionary and validate labels and mappings. | `approved_variable_dictionary_b` |
| 7 | Approval Processing (1.5.2) | Data Analyst | Execute approval process for the selected use case and Codebook version. | `hrs_use_case_approved_variables` |
| 8 | Cohort Coverage (1.5.3) | Data Analyst | Execute coverage evaluation and review HAS_DATA and NO_DATA results. | `hrs_use_case_approved_cohort_coverage` |
| 9 | Technical Sign-Off | Data Analyst | Record review metadata and sign off against the Use Case Document. | Signed-Off Metadata |
| 10 | Coverage Confirmation | End User | Confirm acceptance of coverage limitations. | Acknowledged Coverage |

---

# Phase 4 – Agent Drafting

| # | Step | Role | Activity | Output |
|---|---|---|---|---|
| 11 | Set Use-Case Widgets | Data Analyst | Configure `use_case_name` and `codebook_version` values in all required notebooks. | Configured Notebooks |
| 12 | Open & Run HRS Agent | Data Engineer | Execute the runner notebook and validate metadata grounding. | Scoped and Verified Agent Session |
| 13 | Read Routing Report | Data Engineer | Review routing results and determine whether deployment is necessary. | Routing Decision |
| 14 | Agent Run | Agent | Generate routing reports, DDL/DML drafts, and metadata recommendations. | Draft Artifacts and Audit Trail |
| 15 | Review Drafts | Data Engineer | Review and approve or reject generated drafts. | Draft Decisions |

---

# Phase 5 – Deploy & Verify

| # | Step | Role | Activity | Output |
|---|---|---|---|---|
| 16 | Apply Drafts | Data Engineer | Commit approved scripts through the Git workflow. | Committed Scripts |
| 17 | Execute DDL + DML | Data Engineer | Execute approved repository scripts against the target catalog. | Updated Silver Tables |
| 18 | Re-Run Agent Routing | Data Engineer → Agent | Validate that all routing gaps are resolved after deployment. | Zero-Gap Routing Report |
| 19 | Verify Loaded Tables | Data Engineer | Execute data quality and validation checks. | Verification Results |
| 20 | Consume Results | End User | Use Gold-layer data products and provide feedback. | Delivered Analysis |
# Phase 6 – Gold Design

| # | Step | Role | Activity | Output / Artifact |
|---|---|---|---|---|
| 21 | Define Gold Deliverables | Data Analyst | Using the Use Case Document's derived measures and grain definitions, determine required aggregates (for example, `condition_count`, mean/min/max calculations), target grain (for example, cohort × wave), required dimensions, and display labels. | Gold Design Specification |
| 22 | Check Existing Gold Coverage | Data Engineer | Determine whether an existing Gold fact already satisfies the analysis requirements (similar to `fact_hrs_bmi_stats` for BMI analytics). Apply idempotency principles: reuse or refresh existing Gold assets before creating new ones. | Build vs. Reuse Decision |

---

# Phase 7 – Gold Drafting

| # | Step | Role | Activity | Output / Artifact |
|---|---|---|---|---|
| 23 | Agent Gold Routing Run | Data Engineer → Agent | Execute Run HRS Agent using the same `use_case_name`. Route against the live Gold schema and generate either a coverage report for existing Gold facts or aggregate DDL/DML drafts for coverage gaps. | Coverage Report or Draft Artifacts |
| 24 | Review Gold Drafts | Data Engineer | Validate aggregate grain, presentation-ready measures, numeric code preservation, label decoding through dimensions, `active = TRUE` filtering, and star-schema joins through `vw_dim_wave` and `vw_dim_cohort`. | Draft Decisions |
| 25 | Codebook Label Cross-Check | Agent | Execute `generate_codebook_label_dimension` to validate curated dimension labels (for example, cohort labels) against Codebook definitions before report publication. | Label Verification |

---

# Phase 8 – Gold Deploy & Verify

| # | Step | Role | Activity | Output / Artifact |
|---|---|---|---|---|
| 26 | Apply Drafts | Data Engineer | Write approved DDL/DML artifacts into `sql/gold_ddl/` and `sql/gold_dml/`. Complete feature branch → pull request → CI → merge workflow. | Committed Scripts |
| 27 | Execute DDL + DML | Data Engineer | Execute repository-based Gold scripts. Perform `TRUNCATE` and `INSERT` operations to populate Gold aggregates from Silver facts and dimensions. | Gold Facts Loaded |
| 28 | Verify Aggregates | Data Engineer | Independently recompute aggregates from Silver outputs and compare against Gold results. Validate row counts, expected groupings, and label decoding. | Verification Results |
| 29 | Consume Results | End User | Consume dashboards, reports, and analytics built on the Gold star schema. Feedback may initiate a new use case cycle. | Delivered Analysis |

---

# Gold Layer Workflow Overview

| Phase | # | Step | Role | Activity | Output |
|---------|---|---------|---------|---------|---------|
| 6. Gold Design | 21 | Define Gold Deliverables | Data Analyst | Define required Gold-level aggregates, grain, dimensions, and display labels from the Use Case Document. | Gold Design Specification |
| 6. Gold Design | 22 | Check Existing Gold Coverage | Data Engineer | Determine whether existing Gold assets can support the use case before creating new structures. | Build vs. Reuse Decision |
| 7. Gold Drafting | 23 | Agent Gold Routing Run | Data Engineer → Agent | Generate Gold coverage reports and draft aggregate DDL/DML artifacts where gaps exist. | Coverage Report or Draft Artifacts |
| 7. Gold Drafting | 24 | Review Gold Drafts | Data Engineer | Validate aggregate design standards, joins, filtering, and presentation requirements. | Draft Decisions |
| 7. Gold Drafting | 25 | Codebook Label Cross-Check | Agent | Validate report-facing labels against approved Codebook definitions. | Label Verification |
| 8. Gold Deploy & Verify | 26 | Apply Drafts | Data Engineer | Commit approved Gold DDL/DML scripts through the Git workflow. | Committed Scripts |
| 8. Gold Deploy & Verify | 27 | Execute DDL + DML | Data Engineer | Execute Gold-layer aggregate loading processes from repository files. | Gold Facts Loaded |
| 8. Gold Deploy & Verify | 28 | Verify Aggregates | Data Engineer | Independently validate Gold aggregations and label mappings. | Verification Results |
| 8. Gold Deploy & Verify | 29 | Consume Results | End User | Use dashboards and reports based on Gold data products. | Delivered Analysis |

---

# Gold Layer Review Checklist

| Review Area | Validation Criteria |
|------------|---------------------|
| Aggregate Grain | Matches approved Gold grain (for example, cohort × wave) rather than Silver person-wave grain. |
| Measures | Presentation-ready calculations and appropriate type casts applied. |
| Code Values | Stored as numeric values; report labels provided through dimension joins. |
| Filtering | Includes `active = TRUE` and required NOT NULL filtering. |
| Star Schema Compliance | Uses approved Gold dimensions and star-schema joins. |
| Cohort Labels | Verified through Codebook label validation process. |
| Reusability | Existing Gold facts reused whenever possible. |
| Validation | Gold aggregates independently reconciled back to Silver-level calculations. |

# Silver vs Gold Comparison

| Aspect | Silver Layer | Gold Layer |
|----------|-------------|------------|
| Input Source | Bronze wide table (~19,880 columns) | Verified Silver facts (person-wave grain) |
| Grain | Person-wave model using hub, dimension, and fact tables | Aggregated facts keyed to Gold dimensions |
| Approval Chain | Full governance workflow: Dictionary → Approval Processing (1.5.2) → Cohort Coverage (1.5.3) → Technical Sign-Off | No additional approval workflow. Gold assets are derived only from already-approved Silver variables. |
| New Artifacts per Use Case | Silver DDL/DML generated only when approved variables have no existing CDM home | Gold DDL/DML generated only for new analytical requirements. Shared dimensions (for example, cohort and wave) are reused. |
| Primary Focus | Data engineering concerns: source mapping, data types, unpivoting, business keys, foreign keys, and CDM routing | Semantic modeling concerns: measures, aggregations, labels, reporting structures, and presentation readiness |

---

# Layer Responsibilities

| Category | Silver Layer Responsibilities | Gold Layer Responsibilities |
|------------|------------------------------|-----------------------------|
| Data Integration | Convert RAND HRS source variables into a normalized CDM structure | Consume approved Silver facts and dimensions |
| Variable Mapping | Map source variables to fact attributes | Use existing approved Silver variables |
| Routing | Determine CDM homes for approved variables | Determine analytical destinations for derived measures |
| Data Modeling | Hub/Dimension/Fact design | Star-schema aggregate design |
| Measures | Store detailed observations at respondent-wave level | Create analytical measures and aggregated calculations |
| Metadata Validation | Codebook validation, cohort coverage validation, approval tracking | Label validation and presentation verification |
| Deployment Artifacts | Silver DDL and DML scripts | Gold DDL and DML scripts |
| Consumer Audience | Data Engineers and Data Analysts | Business Analysts, End Users, Dashboards, and Reports |

---

# Workflow Focus by Layer

| Layer | Goal | Typical Outputs |
|---------|---------|---------|
| Silver | Create trusted, approved, reusable business facts from RAND HRS source data | `fact_health`, `fact_demographics`, dimensions, routing metadata, coverage reports |
| Gold | Create business-ready analytical products from approved Silver facts | Aggregate fact tables, KPIs, dashboards, reports, cohort-level and trend analyses |

---

# Key Principle

| Silver | Gold |
|---------|---------|
| Answers: "Where should this source variable live?" | Answers: "How should this approved data be analyzed and presented?" |
| Concerned with data acquisition and standardization | Concerned with business meaning and reporting |
| Variable-centric | Analysis-centric |
| Creates reusable facts | Creates reusable insights |
| Requires approval workflow | Inherits approval from Silver outputs |

---

# Design Philosophy

| Silver Layer | Gold Layer |
|-------------|------------|
| Build once, reuse many times. | Analyze many ways from the same trusted facts. |
| Normalize data and enforce standards. | Optimize for reporting and business consumption. |
| Owns source-data governance. | Owns analytical presentation and interpretation. |
| Focus on completeness and traceability. | Focus on usability and insight generation. |


